# Chapter 8: Inflation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch08-inflation.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every
> interactive figure has live menus, buttons or sliders: change one and the figure is drawn again
> on the spot. On the book website the same figures come with one precomputed slider or a few
> buttons. Each section ends with a short **Your turn** task, and you can try it in the empty code
> cell below the task.

## Learning goals

Inflation is the growth rate of the overall price level. By the end of this chapter you will be able to:

- measure inflation from a price index, and convert a price from one year into the dollars of another
  year with the consumer price index (CPI);
- solve the **quantity theory of money** for the price level, $P^* = \bar{M}\bar{V}/\bar{Y}$, and for
  inflation, $\pi^* = \bar{g}_M - \bar{g}_Y$;
- test the quantity theory with 150 years of U.S. data and with 165 economies;
- compute **real interest rates** with the **Fisher equation**, $i = R + \pi$;
- explain why very high inflation is usually a **fiscal** problem: governments print money to pay their bills;
- rebuild a price index from inflation rates, compute average growth rates over periods, use ratio scales
  with clear tick labels, and spot suspicious numbers in a data set.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook
figure each notebook figure replicates.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 8.1 The inflation rate in the United States | data | inflation rate or price level (buttons); Table 8.1 and a price converter |
| Figure 2 | 8.2 The percentage of countries experiencing high inflation | data | the threshold for "high" inflation (menu) |
| Figure 3 | 8.3 Money growth and inflation in the United States | data | the length of the periods: 5, 10 or 20 years (menu); Table 8.2 recomputed |
| Figure 4 | 8.4 Money growth and inflation around the world | data | real GDP growth in the quantity-theory line (slider); the period (buttons) |
| Figure 5 | 8.5 Real and nominal interest rates in the United States | data | — (click "Inflation" in the legend to show it) |
| Figure 6 | 8.6 Hyperinflations in Argentina, Brazil, and Russia | data | linear or ratio scale (buttons) |
| Figure 7 | 8.7 High inflation in Mexico, Nigeria, and Venezuela | data | a fourth country to compare (menu) |

The textbook's tables are rebuilt where they belong: Table 8.1 (the CPI) in Figure 1, Table 8.2 (measures
of money) in Figure 3, and Table 8.3 (the four equations of the quantity theory) in "The model in code".

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `pd` for pandas, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                    # numbers and arrays
import pandas as pd                   # data tables
import plotly.express as px           # quick charts from data tables
import plotly.graph_objects as go     # charts built piece by piece
import plotly.io as pio
from ipywidgets import interact, FloatSlider, ToggleButtons, Dropdown

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
ORANGE = "#e0a33b"                              # a fourth color for the country comparisons
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch08/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each model figure gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

The chapter's model is the **quantity theory of money**. Let $M_t$ be the amount of money, $V_t$ the
**velocity** of money (how many times a year each dollar is used to buy goods), $P_t$ the price level and
$Y_t$ real GDP. The theory has four equations (Table 8.3 of the textbook):

| Equation | Meaning |
|---|---|
| $M_t V_t = P_t Y_t$ | the quantity equation (8.1): money spent = nominal GDP |
| $Y_t = \bar{Y}_t$ | real GDP comes from the growth model (the **classical dichotomy**) |
| $V_t = \bar{V}$ | velocity is constant |
| $M_t = \bar{M}_t$ | the central bank chooses the money supply |

Only the price level is left for the quantity equation to pin down:

$$P_t^* = \frac{\bar{M}_t \bar{V}}{\bar{Y}_t} \qquad (8.2)$$

In growth rates, the growth of a product is the sum of the growth rates, so
$\bar{g}_M + \bar{g}_V = g_P + \bar{g}_Y$ (8.3). With constant velocity, $\bar{g}_V = 0$, and inflation
$\pi = g_P$ is

$$\pi^* = \bar{g}_M - \bar{g}_Y \qquad (8.4)$$

Inflation also links the two kinds of interest rates. The **nominal** rate $i$ is paid in dollars, the
**real** rate $R$ in goods, and the **Fisher equation** says

$$i = R + \pi \qquad (8.5) \qquad\qquad R = i - \pi \qquad (8.6)$$

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the *arguments*) and
> gives back a result with `return`. Writing `g_V=0.0` gives an argument a *default value*, used
> whenever we do not pass one. All rates here are in percent.

In [3]:
def price_level(M, V, Y):
    """Equation (8.2): the price level that solves the quantity equation M V = P Y."""
    return M * V / Y


def qt_inflation(g_M, g_Y, g_V=0.0):
    """Equations (8.3)-(8.4): inflation = money growth + velocity growth - real GDP growth (percent)."""
    return g_M + g_V - g_Y


def nominal_rate(R, pi):
    """Equation (8.5), the Fisher equation: the nominal interest rate i = R + pi (percent)."""
    return R + pi


def real_rate(i, pi):
    """Equation (8.6): the real interest rate R = i - pi (percent)."""
    return i - pi

A quick check of the functions with the textbook's examples.

> **Python note.** An *f-string* such as `f"V = {1000 / 200:.0f}"` inserts a value into a piece of text;
> `:.0f` means "no decimals" and `:.1f` "one decimal".

In [4]:
print(f"Nominal GDP 1,000 and money 200: velocity V = {1000 / 200:.0f}")
print(f"M = 200, V = 5, Y = 500:  price level P* = {price_level(200, 5, 500):.1f}")
print(f"Twice the money, M = 400: price level P* = {price_level(400, 5, 500):.1f}")
print(f"Money growth 7%, real GDP growth 4%:  inflation = {qt_inflation(7, 4):.0f}%")
print(f"Money growth 10%, real GDP growth 4%: inflation = {qt_inflation(10, 4):.0f}%")
print(f"Real rate 5%, inflation 3%: nominal rate i = {nominal_rate(5, 3):.0f}%")
machine_return = 5 + (100 * 1.03 - 100)         # output worth 5, plus the machine's price gain
print(f"A 100-dollar machine earns {machine_return:.0f} dollars: a nominal return of {machine_return / 100:.0%}")

Nominal GDP 1,000 and money 200: velocity V = 5
M = 200, V = 5, Y = 500:  price level P* = 2.0
Twice the money, M = 400: price level P* = 4.0
Money growth 7%, real GDP growth 4%:  inflation = 3%
Money growth 10%, real GDP growth 4%: inflation = 6%
Real rate 5%, inflation 3%: nominal rate i = 8%
A 100-dollar machine earns 8 dollars: a nominal return of 8%


Doubling the money supply doubles the price level and changes nothing real: this is the **neutrality of
money**. And raising money growth from 7 to 10 percent raises inflation from 3 to 6 percent, one for one.

Equation (8.4) is a rule for growth rates, and it is exact only for small changes. To see how close it is,
we let money and real GDP grow year by year and solve equation (8.2) in every year.

> **Python note.** `for t in range(years + 1)` repeats the indented lines for t = 0, 1, ..., years.
> `rows.append([...])` adds one row to a list, and `pd.DataFrame(rows, columns=[...])` turns the list into
> a table. `.pct_change()` gives the change from the previous row as a fraction.

In [5]:
def simulate_prices(g_M=7.0, g_Y=4.0, V=5.0, years=5):
    """Equation (8.2) year by year: money grows at g_M percent and real GDP at g_Y percent."""
    M, Y = 200.0, 500.0
    rows = []
    for t in range(years + 1):
        rows.append([t, M, Y, price_level(M, V, Y)])
        M = M * (1 + g_M / 100)
        Y = Y * (1 + g_Y / 100)
    path = pd.DataFrame(rows, columns=["year", "M", "Y", "P"])
    path["inflation"] = 100 * path.P.pct_change()
    return path


simulate_prices().round(3)

,year,M,Y,P,inflation
0,0,200.000,500.000,2.000,NaN
1,1,214.000,520.000,2.058,2.885
2,2,228.980,540.800,2.117,2.885
3,3,245.009,562.432,2.178,2.885
4,4,262.159,584.929,2.241,2.885
5,5,280.510,608.326,2.306,2.885


Inflation is 2.885 percent in every year, not exactly 3: the exact rule is $1.07/1.04 - 1$. For the rates
we meet in the United States the shortcut $\pi \approx g_M - g_Y$ is very close. For the hyperinflations
of Figures 6 and 7 it is not, but the message is the same: more money growth, more inflation.

## Figure 1: The inflation rate in the United States

**Goal.** Follow U.S. inflation since 1960, and see how inflation rates add up to a much higher price level.

If $P_t$ is the price level in year $t$, the inflation rate is the percentage change
$(P_{t+1} - P_t)/P_t$. Here the price level is the **consumer price index** (CPI), the price of a basket
of consumer goods, averaged over the twelve months of each year.

> **Python note.** `pd.read_csv(...)` reads a data file into a *DataFrame*, a table with named columns.
> `.head()` shows its first five rows.

In [6]:
us = pd.read_csv(DATA_URL + "fig8-1.csv")
us.head()

,year,inflation_cpi
0,1960,1.492
1,1961,1.070
2,1962,1.176
3,1963,1.256
4,1964,1.322


Inflation is the growth rate of the price level, so we can also go back from inflation to the price level.
Start at 100 in 1959 and multiply by $(1 + \pi_t/100)$ year after year.

> **Python note.** `.cumprod()` is the *cumulative product*: its first value is the first number, its
> second value the product of the first two numbers, and so on. `.tail()` shows the last five rows.

In [7]:
us["price_level"] = 100 * (1 + us.inflation_cpi / 100).cumprod()
us.tail()

,year,inflation_cpi,price_level
61,2021,4.681,929.617086
62,2022,7.991,1003.902787
63,2023,4.127,1045.333855
64,2024,2.952,1076.192111
65,2025,2.634,1104.539011


The chart function draws either view. The price level is shown on a **ratio scale**, where equal
distances mean equal *percentage* changes, so a steeper line means faster inflation.

> **Python note.** `if ... else` runs one block of code or the other. `px.area` draws a line and fills
> the area below it. With `log_y=True` the vertical axis is a ratio scale; `tickvals` chooses the numbers
> printed on it, and the `range` of a ratio axis is given in powers of 10 (`np.log10(80)`).

In [8]:
PRICE_TICKS = [100, 200, 400, 800, 1600]


def us_inflation(view="inflation rate"):
    """Figure 8.1: U.S. CPI inflation, or the price level it adds up to (1959 = 100, ratio scale)."""
    by_year = us.set_index("year").inflation_cpi
    if view == "inflation rate":
        fig = px.area(us, x="year", y="inflation_cpi", hover_name="year",
                      labels={"year": "Year", "inflation_cpi": "Inflation rate (percent)"},
                      title=f"U.S. inflation peaked at {by_year.max():.1f}% in {by_year.idxmax()} "
                            f"and reached {by_year[2022]:.1f}% in 2022")
        fig.update_traces(line=dict(color=BLUE, width=2))
        fig.add_hline(y=0, line_dash="dash", line_color=GRAY)
        fig.update_yaxes(range=[-2, 15])
    else:
        factor = us.price_level.iloc[-1] / 100
        fig = px.line(us, x="year", y="price_level", log_y=True, hover_name="year",
                      labels={"year": "Year", "price_level": "Price level (1959 = 100, ratio scale)"},
                      title=f"Prices rose {factor:.1f}-fold from 1959 to {us.year.iloc[-1]}; steepest in the 1970s")
        fig.update_traces(line=dict(color=BLUE, width=3))
        fig.update_yaxes(tickvals=PRICE_TICKS, range=[np.log10(80), np.log10(1500)])
    return fig

> **Python note.** `interact` calls the function and redraws its figure every time you change a widget.
> `ToggleButtons` shows one button per option, and the function receives the text of the chosen button.
> The `;` at the end only hides an extra line of text.

In [9]:
interact(us_inflation, view=ToggleButtons(options=["inflation rate", "price level"], description="show"));

In [10]:
# @title Website version: Figure 1 with buttons for the two views
rate_fig, level_fig = us_inflation("inflation rate"), us_inflation("price level")
fig = go.Figure(layout=rate_fig.layout)
fig.add_trace(rate_fig.data[0])
fig.add_trace(level_fig.data[0].update(visible=False))
rate_view = {"title.text": rate_fig.layout.title.text, "yaxis.type": "linear", "yaxis.range": [-2, 15],
             "yaxis.tickvals": None, "yaxis.title.text": rate_fig.layout.yaxis.title.text,
             "shapes": [s.to_plotly_json() for s in rate_fig.layout.shapes]}
level_view = {"title.text": level_fig.layout.title.text, "yaxis.type": "log",
              "yaxis.range": [np.log10(80), np.log10(1500)], "yaxis.tickvals": PRICE_TICKS,
              "yaxis.title.text": level_fig.layout.yaxis.title.text, "shapes": []}
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0.01, y=0.98, xanchor="left",
                                    yanchor="top", buttons=[
    dict(label="inflation rate", method="update", args=[{"visible": [True, False]}, rate_view]),
    dict(label="price level", method="update", args=[{"visible": [False, True]}, level_view])])])
fig

**What to notice.**

- Inflation was low in the early 1960s, rose through the late 1960s and 1970s (the "Great Inflation"), and
  peaked at 13.5 percent in 1980.
- It fell sharply in the early 1980s, and from 1992 to 2020 it stayed below 4 percent. The Covid-19 years
  brought it back to 8.0 percent in 2022.
- In 2009 the rate was slightly negative: prices fell. That is **deflation**.
- Switch to the price level: prices are about 11 times their 1959 level. The line never falls for long,
  and it is steepest in the 1970s.

### Key numbers: the Great Inflation and after

> **Python note.** `set_index("year")` uses the years as row labels, so `rate.loc[1960:1964]` keeps the
> rows from 1960 to 1964 and `rate[1980]` picks one number.

In [11]:
rate = us.set_index("year").inflation_cpi
print(f"Average 1960-1964:  {rate.loc[1960:1964].mean():.2f}%")
print(f"Peak:               {rate.max():.2f}% in {rate.idxmax()}")
print(f"Average 2000-2019:  {rate.loc[2000:2019].mean():.2f}%")
print(f"2009: {rate[2009]:.2f}%   2022: {rate[2022]:.2f}%   {rate.index[-1]}: {rate.iloc[-1]:.2f}%")

Average 1960-1964:  1.26%
Peak:               13.50% in 1980
Average 2000-2019:  2.17%
2009: -0.32%   2022: 7.99%   2025: 2.63%


The textbook's numbers hold: about 1.3 percent a year in the early 1960s ("on the order of 2 percent"), a peak
of 13.50 percent in 1980, about 2 percent in 2000–2019, and 7.99 percent in 2022. By 2025 inflation was down
to 2.63 percent.

### Table 8.1 and the case study "How much is that?"

How do we compare 27 cents in 1950 with \$3.50 in 2022? A longer CPI series, from MeasuringWorth, starts in
1774. Table 8.1 rescales it so that 2022 = 100; we add a column with the latest year, 2025, as the base.

In [12]:
cpi = pd.read_csv(DATA_URL + "us_cpi.csv").set_index("year").cpi
years = [1900, 1930, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2022, 2025]
table = pd.DataFrame({"CPI, 2022 = 100": 100 * cpi[years] / cpi[2022],
                      "CPI, 2025 = 100": 100 * cpi[years] / cpi[2025]})
table["Textbook (2022 = 100)"] = [2.78, 5.71, 8.23, 10.12, 13.27, 28.15, 44.66, 58.84, 74.51, 100.00, None]
table.round(2)

,"CPI, 2022 = 100","CPI, 2025 = 100",Textbook (2022 = 100)
year,,,
1900,2.78,2.53,2.78
1930,5.71,5.19,5.71
1950,8.23,7.48,8.23
1960,10.12,9.20,10.12
1970,13.27,12.06,13.27
1980,28.15,25.59,28.15
1990,44.66,40.60,44.66
2000,58.84,53.49,58.84
2010,74.51,67.73,74.51


Every cell of the textbook's table is reproduced. To move a price from one year to another, multiply by the
ratio of the two price levels: $0.27 \times 100/8.23$ for 1950 to 2022.

In [13]:
def in_dollars_of(price, year, to_year=2025):
    """Table 8.1 case study: the value of `price` dollars of `year`, in dollars of `to_year`."""
    return price * cpi[to_year] / cpi[year]


print(f"27 cents of 1950 = {in_dollars_of(0.27, 1950, 2022):.2f} dollars of 2022 = {in_dollars_of(0.27, 1950):.2f} dollars of 2025")
print(f"Prices rose {100 * (cpi[2025] / cpi[2022] - 1):.1f}% from 2022 to 2025")
print(f"A 10,000-dollar pension of 1970 bought in 1983 what {in_dollars_of(10000, 1983, 1970):,.0f} dollars bought in 1970")

27 cents of 1950 = 3.28 dollars of 2022 = 3.61 dollars of 2025
Prices rose 10.0% from 2022 to 2025
A 10,000-dollar pension of 1970 bought in 1983 what 3,900 dollars bought in 1970


Gasoline at 27 cents in 1950 cost \$3.28 in 2022 dollars, just a little less than its 2022 price. The last line
is Mr. Gambino from the textbook's Section 8.4: a fixed pension of \$10,000 set in 1970 had lost 61 percent of
its value by 1983, and was worth about \$3,900 of 1970 dollars (the textbook says 40 percent, \$4,000).

**Your turn.** What were 100 dollars of 1900 worth in dollars of 2025? And by how much did prices rise on
average per year between 1980 and 2022?

<details><summary>Hint</summary>

Use `in_dollars_of(100, 1900)`. For the average annual rate, use $(P_{2022}/P_{1980})^{1/42} - 1$ with
`cpi[2022]` and `cpi[1980]`.

</details>

In [14]:
# Your turn: try it here

## Figure 2: The percentage of countries experiencing high inflation

**Goal.** See that high inflation was common around the world in the 1970s–1990s, and rare since 2000.

For each year, the textbook counts the share of countries with inflation above 25 percent. The file has this
share, and the number of countries with data in each year.

In [15]:
high = pd.read_csv(DATA_URL + "fig8-2.csv")
high.head()

,year,share_above_25_pct,n_countries
0,1960,3.175,63
1,1961,0.000,65
2,1962,1.493,67
3,1963,2.941,68
4,1964,5.556,72


To try other thresholds we need the inflation rate of every country in every year. A wider file has them.

In [16]:
panel = pd.read_csv(DATA_URL + "inflation_panel.csv")
panel.head()

,iso3,country,year,inflation,source
0,AFG,Afghanistan,1970,25.51,GID
1,AFG,Afghanistan,1971,25.51,GID
2,AFG,Afghanistan,1972,-12.52,GID
3,AFG,Afghanistan,1973,-10.68,GID
4,AFG,Afghanistan,1974,10.23,GID


The column `source` says where each number comes from. `WDI` rows are the World Bank's World Development
Indicators, the data behind the textbook's figure. `GID` rows fill gaps from the World Bank's Global Inflation
Database; we need them in Figures 6 and 7. To redraw Figure 8.2 we keep only the WDI rows.

> **Python note.** `panel[panel.source == "WDI"]` keeps the rows where the condition is true.
> `wdi.inflation > threshold` is a column of True/False values; its `.mean()` is the share of True values,
> because True counts as 1 and False as 0. `groupby(wdi.year)` computes this share separately for each year.

In [17]:
wdi = panel[panel.source == "WDI"]


def high_inflation_share(threshold=25):
    """Figure 8.2: the percentage of reporting economies with inflation above `threshold` percent."""
    above = wdi.inflation > threshold
    share = (100 * above.groupby(wdi.year).mean()).reset_index(name="share")
    peak = share.loc[share.share.idxmax()]
    fig = px.area(share, x="year", y="share", hover_name="year",
                  labels={"year": "Year", "share": "Fraction of countries (percent)"},
                  title=f"Inflation above {threshold}%: the share of countries peaked at {peak.share:.1f}% in {peak.year:.0f}")
    fig.update_traces(line=dict(color=TEAL, width=2))
    fig.update_yaxes(range=[0, 90])
    return fig

Before using it, we check that the threshold of 25 percent reproduces the file of Figure 8.2.

In [18]:
share_25 = 100 * (wdi.inflation > 25).groupby(wdi.year).mean()
print(f"Largest difference from fig8-2.csv: {(share_25.values - high.share_above_25_pct.values).max():.3f} points")

Largest difference from fig8-2.csv: 0.000 points


> **Python note.** A `Dropdown` is a menu: the function receives the option you pick.

In [19]:
interact(high_inflation_share,
         threshold=Dropdown(options=[10, 25, 50, 100], value=25, description="inflation above (percent)", style=WIDE));

In [20]:
# @title Website version: Figure 2 with a precomputed slider for the threshold
slider_figure(high_inflation_share, "threshold", [10, 25, 50, 100], start=25, label="inflation above (percent)")

**What to notice.**

- With the textbook's threshold of 25 percent, the share was small in the 1960s, jumped in 1974, stayed
  high in the 1980s and peaked in 1994, when almost one country in three had high inflation.
- After the mid-1990s the share collapsed. The Covid-19 years brought a small rise, and by 2025 it was down
  to 3 percent.
- Choose 10 percent: in 1974, most countries had double-digit inflation. Choose 100 percent: inflation of
  that size never hit more than about one country in nine, and it peaked in the early 1990s.

### Key numbers: high inflation over time

In [21]:
print(f"1960s: at most {share_25.loc[1960:1969].max():.1f}% of countries")
print(f"Peak: {share_25.max():.1f}% in {share_25.idxmax()}")
print(f"2002-2025: at most {share_25.loc[2002:].max():.1f}% (in {share_25.loc[2002:].idxmax()});  2025: {share_25[2025]:.1f}%")
print(f"Countries with data: {high.n_countries.iloc[0]} in 1960, {high.n_countries.max()} at most, {high.n_countries.iloc[-1]} in 2025")

1960s: at most 5.6% of countries
Peak: 31.1% in 1994
2002-2025: at most 7.9% (in 2023);  2025: 3.0%
Countries with data: 63 in 1960, 192 at most, 169 in 2025


The textbook's description holds: below 6 percent in the 1960s, 31.1 percent in 1994 ("almost one out of every
three countries"), and below 5 percent in every year from 2002 to 2021. The Covid-19 years broke the pattern, with a
peak of 7.9 percent in 2023.
Notice also that the number of countries with data more than doubles over the period, so the early shares are
based on fewer countries.

**Your turn.** In which year was the share of countries with inflation above 10 percent highest, and how high
was it? Compare it with the peak for 25 percent.

<details><summary>Hint</summary>

Compute `share_10 = 100 * (wdi.inflation > 10).groupby(wdi.year).mean()` and use `.max()` and `.idxmax()`.

</details>

In [22]:
# Your turn: try it here

## Figure 3: Money growth and inflation in the United States

**Goal.** Test the main prediction of the quantity theory with U.S. history: periods of fast money growth
should be periods of high inflation.

Each point is one decade. Money is **M2** (currency, checking accounts, savings deposits and money market
accounts; see Table 8.2 below). Both growth rates are average annual rates from the first year of a decade
to the first year of the next:

$$g = \left(\frac{X_{d+10}}{X_d}\right)^{1/10} - 1$$

In [23]:
decades = pd.read_csv(DATA_URL + "fig8-3.csv")
decades.tail()

,year,decade,end_year,money_growth,inflation
11,1980,1980s,1990,7.660,4.724
12,1990,1990s,2000,4.049,2.796
13,2000,2000s,2010,6.078,2.389
14,2010,2010s,2020,7.425,1.728
15,2020,2020s (2020-25),2025,4.414,4.462


The last row covers only 2020–2025, a decade that is not over yet. To choose other period lengths, we load
the yearly M2 and CPI series behind the file and compute the growth rates ourselves.

> **Python note.** `range(1870, 2025, years)` counts 1870, 1870 + years, ... up to 2024: the first year of
> each period. `min(a, b)` is the smaller number, so the last period stops in 2025. `x ** (1 / n)` is the
> n-th root.

In [24]:
money = pd.read_csv(DATA_URL + "us_money_prices.csv").set_index("year")


def growth(x, start, end):
    """Average annual growth of x from start to end, in percent."""
    return 100 * ((x[end] / x[start]) ** (1 / (end - start)) - 1)


def periods(years=10):
    """Figure 8.3: average money growth and inflation in periods of `years` years since 1870."""
    rows = []
    for start in range(1870, 2025, years):
        end = min(start + years, 2025)
        if years == 10 and end - start == 10:
            label = f"{start}s"
        else:
            label = f"{start}–{end}"
        kind = "full period" if end - start == years else "unfinished period"
        rows.append([label, start, end, growth(money.m2, start, end), growth(money.cpi, start, end), kind])
    return pd.DataFrame(rows, columns=["period", "start", "end", "money_growth", "inflation", "kind"])


check = periods(10)
print(f"Largest difference from fig8-3.csv: {(check.money_growth - decades.money_growth).abs().max():.3f} (money), "
      f"{(check.inflation - decades.inflation).abs().max():.3f} (inflation)")

Largest difference from fig8-3.csv: 0.000 (money), 0.000 (inflation)


The quantity theory says $\pi = g_M + g_V - g_Y$. If velocity were constant and real GDP did not grow, every
point would lie on the 45-degree line $\pi = g_M$. Real GDP does grow, so points should lie **below** that line,
by about $g_Y - g_V$. The chart draws both lines; the lower one is shifted down by the average gap in the data.

> **Python note.** `px.scatter(..., symbol="kind")` gives each kind of point its own marker, and
> `symbol_map` chooses the markers: full circles for full periods, an open circle for the unfinished one.
> `.corr()` is the *correlation* of two columns: 1 for points on a rising straight line, 0 for no relation.

In [25]:
KEY_DECADES = ["1870s", "1930s", "1940s", "1970s", "1980s", "2010s", "2020–2025"]


def money_and_prices(years=10):
    """Figure 8.3: U.S. money growth against inflation, one point per period of `years` years."""
    table = periods(years)
    gap = (table.money_growth - table.inflation).mean()
    r = table.money_growth.corr(table.inflation)
    labels = table.period.where(table.period.isin(KEY_DECADES) | (years == 20), "")   # hover shows the rest
    fig = px.scatter(table, x="money_growth", y="inflation", symbol="kind", text=labels, hover_name="period",
                     symbol_map={"full period": "circle", "unfinished period": "circle-open"},
                     labels={"money_growth": "Money growth (percent per year)", "inflation": "Inflation rate (percent per year)",
                             "kind": ""},
                     title=f"{years}-year periods: correlation {r:.2f}, average gap g_M − π = {gap:.1f} points")
    fig.update_traces(marker=dict(color=TEAL, size=11), textposition="top center", textfont=dict(color=BLUE, size=11))
    g_M = np.array([-3, 18])
    fig.add_trace(go.Scatter(x=g_M, y=g_M, mode="lines", name="π = g_M (45-degree line)",
                             line=dict(color=GRAY, dash="dot")))
    fig.add_trace(go.Scatter(x=g_M, y=qt_inflation(g_M, gap), mode="lines", name=f"π = g_M − {gap:.1f}",
                             line=dict(color=GREEN, width=3, dash="dash")))
    fig.add_hline(y=0, line_dash="dash", line_color=GRAY, line_width=1)
    fig.update_xaxes(range=[-3, 18])
    fig.update_yaxes(range=[-5, 17])
    fig.update_layout(legend=dict(orientation="h", y=-0.15, yanchor="top"))
    return fig

In [26]:
interact(money_and_prices,
         years=Dropdown(options=[5, 10, 20], value=10, description="years per period", style=WIDE));

In [27]:
# @title Website version: Figure 3 with a precomputed slider for the length of the periods
slider_figure(money_and_prices, "years", [5, 10, 20], start=10, label="years per period")

**What to notice.**

- With decades, as in the textbook, the 1910s and the 1970s have the fastest inflation and fast money growth.
  The 1920s and 1930s had slow money growth and **deflation**: prices fell.
- The points rise from left to right, roughly parallel to the 45-degree line, and about 4 points below it.
  The quantity theory says this gap is the growth of real GDP minus the growth of velocity.
- With 5-year periods the cloud is wider. Wars and the Great Depression move prices a lot over a few years,
  and the theory is about the long run. Hover over the points to see the periods.
- With 20-year periods there are only 8 points, all with money growth between 5 and 9 percent.

### Table 8.2: measures of the money supply

What counts as money? The narrowest measure is **currency**. The **monetary base** adds the reserves banks
hold at the central bank, **M1** adds checking accounts, and **M2** adds savings and money market accounts.
A monthly file from the Federal Reserve has all four, in billions of dollars.

> **Python note.** `parse_dates=["date"]` turns text such as `"2023-01-01"` into real dates.
> `.loc[[...]]` picks rows by their labels, and `.T` *transposes* the table: rows become columns.
> `{last:%B %Y}` prints a date as "August 2026".

In [28]:
supply = pd.read_csv(DATA_URL + "money_supply.csv", parse_dates=["date"]).set_index("date")
last = supply.index[-1]
table = supply.loc[[pd.Timestamp("2023-01-01"), last]].T
table.columns = ["January 2023", f"{last:%B %Y}"]
table["Textbook (Jan. 2023)"] = [2300, 5300, 19600, 21300]
table.round(0)

,January 2023,August 2026,Textbook (Jan. 2023)
currency,2299.0,2476.0,2300
monetary_base,5329.0,5412.0,5300
m1,19597.0,19991.0,19600
m2,21274.0,23343.0,21300


January 2023 reproduces the textbook's table within its rounding. Something odd stands out: M1 is almost as
large as M2. Look at the spring of 2020:

In [29]:
supply.loc["2020-03":"2020-07"]

,currency,monetary_base,m1,m2
date,,,,
2020-03-01,1838.0,3883.1,4322.9,16034.4
2020-04-01,1891.3,4844.9,4856.4,17064.0
2020-05-01,1931.8,5149.4,16312.5,17931.9
2020-06-01,1958.3,5001.8,16598.6,18180.0
2020-07-01,1981.7,4700.3,16801.3,18331.4


M1 more than triples between April and May 2020, from 4,856 to 16,313 billion dollars. Nobody printed 11
trillion dollars that month: in May 2020 the Federal Reserve changed the definition of M1 and moved savings
deposits into it. A jump like this is a **series break**. Growth rates across the break are meaningless, which
is one reason Figure 3 uses M2, whose definition did not change.

### Key numbers: is velocity constant?

The quantity theory assumes a constant velocity. The quantity equation (8.1) lets us measure it:
$V = PY/M$, nominal GDP divided by money. A small annual file has nominal GDP, currency and M2.

In [30]:
yearly = pd.read_csv(DATA_URL + "us_money_annual.csv").set_index("year")
yearly["velocity"] = yearly.nominal_gdp / yearly.m2
fig = px.line(yearly.reset_index(), x="year", y="velocity", hover_name="year",
              labels={"year": "Year", "velocity": "Velocity of M2 (nominal GDP / M2)"},
              title="M2 velocity stayed near 1.8 until about 1990, then rose and fell")
fig.update_traces(line=dict(color=BLUE, width=3))
fig.update_yaxes(range=[0, 2.5])
fig.show()
print(yearly.velocity.loc[[1960, 1970, 1980, 1990, 1997, 2007, 2021, 2025]].round(2).to_dict())

{1960: 1.78, 1970: 1.78, 1980: 1.86, 1990: 1.85, 1997: 2.18, 2007: 1.98, 2021: 1.16, 2025: 1.41}


From 1960 to 1990, M2 velocity stayed between about 1.7 and 1.9, close to the textbook's assumption. Then it rose
to 2.18 in 1997 and fell to 1.16 in 2021, when savings piled up during the pandemic. When velocity falls, money
can grow faster than nominal GDP without extra inflation: in equation (8.3) $\bar{g}_V$ is negative.

**Your turn.** Use `periods(20)`. Which 20-year period had the highest inflation, and how fast did money grow
in it? Is the gap $g_M - \pi$ in that period bigger or smaller than the average gap?

<details><summary>Hint</summary>

`periods(20).sort_values("inflation")` sorts the periods from the lowest to the highest inflation. Add a column
with the gap: `table["gap"] = table.money_growth - table.inflation`.

</details>

In [31]:
# Your turn: try it here

## Figure 4: Money growth and inflation around the world

**Goal.** Test the quantity theory across countries: economies where money grows fast should have high inflation.

Each row is one economy. The columns are average annual money growth (broad money) and average annual
inflation, in percent, for two periods: the textbook's 1990–2021 and the latest 1990–2025.

In [32]:
world = pd.read_csv(DATA_URL + "fig8-4.csv")
world.head()

,iso3,country,money_growth_1990_2025,inflation_1990_2025,n_money_1990_2025,n_infl_1990_2025,n_infl_filled_1990_2025,money_growth_1990_2021,inflation_1990_2021,n_money_1990_2021,n_infl_1990_2021,n_infl_filled_1990_2021
0,ABW,Aruba,6.655,2.863,34,35,5,6.943,2.800,32,32,2
1,AFG,Afghanistan,17.313,12.544,15,35,15,17.313,13.643,15,32,15
2,AGO,Angola,200.600,310.970,30,36,1,229.545,347.235,26,32,1
3,ALB,Albania,11.919,14.210,30,36,2,12.786,15.491,27,32,2
4,ARE,United Arab Emirates,11.441,3.166,36,36,18,10.959,3.254,32,32,18


The `n_` columns count the years behind each average, and `n_infl_filled` the years of inflation taken from the
Global Inflation Database because the WDI no longer has them. Three steps of data hygiene were already done when
the file was built: aggregates such as "World" were removed, economies with fewer than 5 years were dropped, and
a few broad-money **series breaks** (a currency reform shows up as money "growth" of over 100,000 percent) were
deleted. One country is new compared with the textbook:

In [33]:
world[world.iso3.isin(["VEN", "COD", "GEO", "BRA", "JPN", "CHE"])][["country", "money_growth_1990_2025", "inflation_1990_2025",
                                                                     "n_money_1990_2025", "n_infl_1990_2025"]]

,country,money_growth_1990_2025,inflation_1990_2025,n_money_1990_2025,n_infl_1990_2025
21,Brazil,215.952,238.733,36,36
28,Switzerland,4.820,1.090,27,36
33,"Congo, Dem. Rep.",560.681,995.432,31,35
53,Georgia,21.589,587.690,30,34
74,Japan,2.130,0.712,36,36
156,"Venezuela, RB",40.714,2613.950,24,35


Venezuela's average inflation of 2,614 percent is not an error: it is the hyperinflation of 2017–2021 (Figure 7),
which is not in the textbook's data. Unlike Venezuela's GDP numbers in earlier chapters, these numbers are real,
so we keep Venezuela and label it.

Inflation ranges from under 1 to over 2,000 percent, so both axes use a ratio scale, as in the textbook. We
label a few economies; hover over any point to see its name.

> **Python note.** `.map(LABELS)` replaces each code with its label from the dictionary `LABELS`; codes that
> are not in it become empty (`NaN`), and `.fillna("")` turns those into empty text. `np.geomspace(a, b, 200)`
> gives 200 numbers from `a` to `b` that are evenly spaced on a ratio scale.

In [34]:
LABELS = {"JPN": "Japan", "CHE": "Switzerland", "DNK": "Denmark", "TUR": "Turkey", "ARG": "Argentina",
          "RUS": "Russia", "GEO": "Georgia", "BRA": "Brazil", "AGO": "Angola", "COD": "Congo, Dem. Rep.",
          "VEN": "Venezuela", "UKR": "Ukraine"}
world["label"] = world.iso3.map(LABELS).fillna("")
POSITIONS = {"COD": "middle left", "BRA": "bottom right", "AGO": "top left"}   # labels that would be cut off or overlap
world["position"] = world.iso3.map(POSITIONS).fillna("middle right")
MONEY_TICKS = [2, 4, 8, 16, 32, 64, 128, 256, 512]
INFLATION_TICKS = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]   # no label in the bottom-left corner


def world_money(g_Y=3.0, period="1990–2025"):
    """Figure 8.4: average money growth and inflation by economy, with the line pi = g_M - g_Y."""
    x, y = "money_growth_" + period.replace("–", "_"), "inflation_" + period.replace("–", "_")
    above = (world[y] > qt_inflation(world[x], g_Y)).sum()
    fig = px.scatter(world, x=x, y=y, text="label", hover_name="country", log_x=True, log_y=True,
                     labels={x: f"Money growth, {period} average (percent, ratio scale)",
                             y: "Inflation rate (percent, ratio scale)"},
                     title=f"π = g_M − g_Y with g_Y = {g_Y:.0f}%: {above} of {len(world)} economies lie above the line")
    fig.update_traces(marker=dict(color=TEAL, size=8), textposition=list(world.position), textfont=dict(color=BLUE, size=11))
    g_M = np.geomspace(g_Y + 0.4, 900, 200)
    fig.add_trace(go.Scatter(x=g_M, y=qt_inflation(g_M, g_Y), mode="lines", name=f"Quantity theory, π = g_M − {g_Y:.0f}",
                             line=dict(color=GREEN, width=3, dash="dash")))
    fig.update_xaxes(tickvals=MONEY_TICKS, range=[np.log10(1.5), np.log10(900)])
    fig.update_yaxes(tickvals=INFLATION_TICKS, range=[np.log10(0.3), np.log10(4000)])
    fig.update_layout(legend=dict(x=0.99, y=0.02, xanchor="right", yanchor="bottom"))
    return fig

Move the slider to change the growth rate of real GDP in the quantity-theory line, and switch between the two periods.

In [35]:
interact(world_money,
         g_Y=FloatSlider(value=3, min=0, max=8, step=1, description="real GDP growth g_Y (percent)", style=WIDE),
         period=ToggleButtons(options=["1990–2025", "1990–2021"], description="period"));

In [36]:
# @title Website version: Figure 4 with a precomputed slider for real GDP growth in the quantity-theory line
slider_figure(world_money, "g_Y", [0, 1, 2, 3, 4, 5, 6, 7, 8], start=3, label="real GDP growth g_Y (percent)")

**What to notice.**

- The economies line up along a rising band: the faster money grows, the higher inflation. Brazil, Angola and
  the Democratic Republic of the Congo have both money growth and inflation in the hundreds of percent.
- Most economies sit in the dense cloud at the lower left, with money growth of about 10–20 percent and inflation
  of about 3–18 percent (the middle half of each column, printed below). Japan and Switzerland are near the bottom.
- With $g_Y = 0$ the line is the 45-degree line: most economies lie below it, because real GDP grows. On a ratio
  scale, subtracting $g_Y$ bends the line down a lot at low money growth and hardly at all at high money growth.
- Georgia and Venezuela lie far above the band, for a statistical reason. Look at the `n_` columns above: their
  money averages cover fewer years than their inflation averages. In the World Bank data, Georgia's money series
  starts in 1996, after its hyperinflation, and Venezuela's stops in 2013, before its hyperinflation. An average
  of inflation over crisis years compared with an average of money growth over calm years is not a fair test.

### Key numbers: how well does the quantity theory fit?

> **Python note.** `np.log10` takes logarithms, the numbers behind a ratio scale. The correlation of the logs
> measures how close the points are to a straight line on the chart.

In [37]:
for period in ["1990_2021", "1990_2025"]:
    g, p = world["money_growth_" + period], world["inflation_" + period]
    print(f"{period}: correlation of the logs {np.log10(g).corr(np.log10(p)):.2f}; "
          f"median gap g_M - pi {(g - p).median():.1f} points; inflation above money growth in {(p > g).sum()} economies")
print("Middle half of money growth:", world.money_growth_1990_2025.quantile([0.25, 0.75]).round(1).tolist(),
      " of inflation:", world.inflation_1990_2025.quantile([0.25, 0.75]).round(1).tolist())
textbook = world.set_index("country").loc[["Denmark", "Switzerland", "Turkiye", "Argentina"]]
textbook[["money_growth_1990_2021", "inflation_1990_2021"]].round(1)

1990_2021: correlation of the logs 0.85; median gap g_M - pi 5.8 points; inflation above money growth in 29 economies
1990_2025: correlation of the logs 0.85; median gap g_M - pi 5.6 points; inflation above money growth in 29 economies
Middle half of money growth: [9.4, 20.8]  of inflation: [3.2, 17.8]


,money_growth_1990_2021,inflation_1990_2021
country,,
Denmark,4.2,1.8
Switzerland,4.8,1.0
Turkiye,46.8,35.3
Argentina,63.6,92.6


The textbook's examples hold in the 1990–2021 averages: Denmark and Switzerland had inflation of 1.8 and 1.0
percent with money growth of about 4–5 percent, while Turkey and Argentina averaged above 25 percent for both.
(The World Bank now spells Turkey "Turkiye".) The correlation of the logs is 0.85 in both periods, and the
median economy has money growth 5.6 to 5.8 points above inflation. With constant velocity this gap would be the growth
rate of real GDP; a gap this large suggests that velocity fell in many economies, as it did in the United States after 1997.

**Your turn.** Which value of $g_Y$ puts about half of the 165 economies above the line? Find it with the slider,
and compare it with the median gap printed above.

<details><summary>Hint</summary>

The title counts the economies above the line. An economy lies above the line when $\pi > g_M - g_Y$, that is
when its gap $g_M - \pi$ is smaller than $g_Y$.

</details>

In [38]:
# Your turn: try it here

## Figure 5: Real and nominal interest rates in the United States

**Goal.** Use the Fisher equation to turn a nominal interest rate into a real one.

The nominal rate is the return on 3-month U.S. Treasury bills, averaged over each year. The real rate cannot be
looked up: we compute it with equation (8.6), $R = i - \pi$, using the CPI inflation of Figure 1.

In [39]:
rates = pd.read_csv(DATA_URL + "fig8-5.csv")
rates.head()

,year,nominal_tbill_3m,inflation_cpi,real_rate
0,1960,2.883,1.492,1.391
1,1961,2.354,1.070,1.284
2,1962,2.773,1.176,1.597
3,1963,3.159,1.256,1.903
4,1964,3.547,1.322,2.225


The file already has a `real_rate` column. Let us check it with our function `real_rate` from "The model in code".

In [40]:
R = real_rate(rates.nominal_tbill_3m, rates.inflation_cpi)
print(f"Largest difference from the file's real rate: {(R - rates.real_rate).abs().max():.3f} points")

Largest difference from the file's real rate: 0.001 points


Before plotting, we give the columns readable names for the legend. Inflation is hidden at first; click it in
the legend to show it. Hover over the chart to read all three numbers of a year and check that $i = R + \pi$.

In [41]:
named = rates.rename(columns={"nominal_tbill_3m": "Nominal rate, i", "real_rate": "Real rate, R = i − π",
                              "inflation_cpi": "Inflation, π"})
negative = (rates.real_rate < 0).sum()
fig = px.line(named, x="year", y=["Nominal rate, i", "Real rate, R = i − π", "Inflation, π"], hover_name="year",
              color_discrete_sequence=[BLUE, GREEN, ORANGE],
              labels={"year": "Year", "value": "Percent", "variable": ""},
              title=f"The real rate is the nominal rate minus inflation: negative in {negative} of {len(rates)} years")
fig.update_traces(line=dict(width=3))
fig.update_traces(selector=dict(name="Inflation, π"), visible="legendonly", line=dict(dash="dot"))
fig.add_hline(y=0, line_dash="dash", line_color=GRAY)
fig.update_yaxes(range=[-10, 16], tickvals=[-5, 0, 5, 10, 15])   # no label in the bottom-left corner
fig.update_layout(hovermode="x unified", legend=dict(orientation="h", y=-0.15, yanchor="top"))
fig.show()

**What to notice.**

- The nominal rate is high when inflation is high. It peaked at just over 14 percent in 1981, a year after
  inflation peaked.
- The real rate was negative in the 1970s, in most years since 2008, and sharply negative in 2021–2022, when
  inflation jumped while the nominal rate was still near zero.
- In 2009 the real rate was **above** the nominal rate. Show inflation in the legend: it was negative that year.

### Key numbers: real rates

In [42]:
r = rates.set_index("year")
print(f"Highest nominal rate: {r.nominal_tbill_3m.max():.1f}% in {r.nominal_tbill_3m.idxmax()}")
print(f"2009: nominal {r.nominal_tbill_3m[2009]:.2f}%, inflation {r.inflation_cpi[2009]:.2f}%, real {r.real_rate[2009]:.2f}%")
print("Years with a negative real rate:", r.index[r.real_rate < 0].tolist())
print(f"Lowest real rate: {r.real_rate.min():.2f}% in {r.real_rate.idxmin()};  {r.index[-1]}: {r.real_rate.iloc[-1]:.2f}%")
print(f"Average real rate: 1960-2022 {r.real_rate.loc[:2022].mean():.2f}%, 1960-2007 {r.real_rate.loc[:2007].mean():.2f}%")

Highest nominal rate: 14.0% in 1981
2009: nominal 0.15%, inflation -0.32%, real 0.47%
Years with a negative real rate: [1974, 1975, 1976, 1977, 1978, 1979, 1980, 2003, 2004, 2005, 2008, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2020, 2021, 2022]
Lowest real rate: -5.97% in 2022;  2025: 1.43%
Average real rate: 1960-2022 0.59%, 1960-2007 1.31%


Most of the textbook's statements hold: a nominal peak of 14.0 percent in 1981, a real rate above the nominal
rate in 2009, negative real rates in 2003 and in every year from 2010 to 2018, and −5.97 percent in 2022. One does
not: the textbook says the real rate averages "about 1.5 percent" over the period, but the plotted series averages
0.59 percent over 1960–2022 (1.31 percent over 1960–2007). Since 2023 the real rate has been positive again.

**Your turn.** Suppose that savers expect inflation to stay at last year's rate. Compute this "expected" real rate,
$i_t - \pi_{t-1}$, for 1981. Is it higher or lower than the real rate in the file? Why might the nominal rate peak
a year after inflation?

<details><summary>Hint</summary>

`r.nominal_tbill_3m[1981] - r.inflation_cpi[1980]`. In 1981 inflation was falling, so last year's inflation was higher
than this year's.

</details>

In [43]:
# Your turn: try it here

## Figure 6: Hyperinflations in Argentina, Brazil, and Russia

**Goal.** See how extreme inflation can get, and how quickly a hyperinflation can end.

A **hyperinflation** is inflation above about 500 percent a year. The file has annual CPI inflation for three
countries that went through one.

In [44]:
hyper = pd.read_csv(DATA_URL + "fig8-6.csv")
hyper = hyper.rename(columns={"argentina": "Argentina", "brazil": "Brazil", "russia": "Russia"})
hyper.head()

,year,Argentina,Brazil,Russia
0,1970,13.587,NaN,NaN
1,1971,34.732,NaN,NaN
2,1972,58.446,NaN,NaN
3,1973,61.249,NaN,NaN
4,1974,23.475,NaN,NaN


Two gaps, compared with the textbook: the World Bank no longer publishes Argentina's inflation before 1970, so the
line starts in 1970; and Argentina before 2018 comes from the Global Inflation Database, because the WDI has dropped
those years. Brazil starts in 1981 and Russia in 1993.

On a linear scale, the peaks of 3,000 percent flatten everything else. A ratio scale shows every level of inflation
at once, but it cannot show zero or negative rates: Argentina's deflation of 2001 (−1.1 percent) simply leaves a gap.

In [45]:
RATE_TICKS = [0.3, 1, 3, 10, 30, 100, 300, 1000, 3000]   # no label in the bottom-left corner


def hyperinflations(scale="linear"):
    """Figure 8.6: inflation in Argentina, Brazil and Russia, on a linear or a ratio scale."""
    a = hyper.set_index("year").Argentina
    fig = px.line(hyper, x="year", y=["Argentina", "Brazil", "Russia"], log_y=(scale == "ratio"),
                  color_discrete_sequence=[BLUE, GREEN, ORANGE],
                  labels={"year": "Year", "value": "Inflation rate (percent)", "variable": "Country"},
                  title=f"Hyperinflations stop fast: Argentina went from {a.max():,.0f}% in {a.idxmax()} to {a[1994]:,.0f}% in 1994")
    fig.update_traces(line=dict(width=3))
    fig.update_traces(selector=dict(name="Brazil"), line=dict(dash="dash"))
    fig.update_traces(selector=dict(name="Russia"), line=dict(dash="dash"))
    if scale == "ratio":
        fig.update_yaxes(tickvals=RATE_TICKS, range=[np.log10(0.1), np.log10(5000)])
    else:
        fig.update_yaxes(range=[-100, 3500])
    fig.update_layout(hovermode="x unified")
    return fig

In [46]:
interact(hyperinflations, scale=ToggleButtons(options=["linear", "ratio"], description="vertical scale"));

In [47]:
# @title Website version: Figure 6 with buttons for the vertical scale
fig = hyperinflations()
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0.01, y=0.98, xanchor="left", yanchor="top", buttons=[
    dict(label="linear scale", method="relayout", args=[{"yaxis.type": "linear", "yaxis.range": [-100, 3500],
                                                         "yaxis.tickvals": None}]),
    dict(label="ratio scale", method="relayout", args=[{"yaxis.type": "log",
                                                        "yaxis.range": [np.log10(0.1), np.log10(5000)],
                                                        "yaxis.tickvals": RATE_TICKS}])])])
fig

**What to notice.**

- Argentina had high inflation for decades, with spikes in 1976 and 1984–1985 before the peak of 1989. Brazil
  peaked in 1990 and again in 1993–1994; Russia's series starts at 874 percent in 1993.
- Each hyperinflation ended within a few years: Argentina was below 5 percent by 1994, Brazil by 1998.
- Switch to the ratio scale. It shows what the linear scale hides: Argentina's inflation climbed again in the
  2010s and 2020s, to 220 percent in 2024, while Brazil's and Russia's stayed low.

### The inflation tax

Why would a government let money grow so fast? Because printing money is one way to pay its bills. The
**government budget constraint** says that spending $G$ is paid with taxes $T$, new borrowing $\Delta B$, or new
money $\Delta M$:

$$G = T + \Delta B + \Delta M \qquad (8.7)$$

The revenue from new money is called **seignorage**, or the **inflation tax**. It is paid by everyone who holds
money: when the price level doubles, a 200-dollar bill buys only what 100 dollars bought before.

In [48]:
def money_printed(G, T, dB):
    """Equation (8.7): the new money the government must issue, Delta M = G - T - Delta B."""
    return G - T - dB


print(f"Spending 25% of GDP, taxes 20%, new borrowing 2%: new money = {money_printed(25, 20, 2):.0f}% of GDP")
print(f"If lenders stop lending (new borrowing 0%):        new money = {money_printed(25, 20, 0):.0f}% of GDP")
print(f"Real value of 200 dollars of currency after prices double: {200 / 2:.0f} dollars of today")

Spending 25% of GDP, taxes 20%, new borrowing 2%: new money = 3% of GDP
If lenders stop lending (new borrowing 0%):        new money = 5% of GDP
Real value of 200 dollars of currency after prices double: 100 dollars of today


In the United States, the inflation tax is tiny. Measured as new currency issued each year, as a percentage of GDP:

> **Python note.** `.diff()` gives the change from the previous row: here, the currency added in each year.

In [49]:
yearly["new_currency"] = 100 * yearly.currency.diff() / yearly.nominal_gdp
print(yearly.new_currency.loc[[1975, 1980, 2000, 2020, 2025]].round(2).to_dict())
print(f"Largest: {yearly.new_currency.max():.2f}% of GDP in {yearly.new_currency.idxmax()}")

{1975: 0.4, 1980: 0.38, 2000: 0.34, 2020: 0.96, 2025: 0.15}
Largest: 0.96% of GDP in 2020


Even in the Great Inflation, new currency brought in about 0.4 percent of GDP, and the largest value since 1960 is
under 1 percent of GDP, in 2020. The textbook reports that countries with high inflation often raise about 5 percent
of GDP this way, and Argentina more than 10 percent in the late 1970s and early 1980s. That is why Thomas Sargent
calls persistent high inflation "a fiscal phenomenon".

**Your turn.** A government spends 30 percent of GDP and collects 22 percent in taxes. How much must it borrow to
issue no new money at all? Check your answer with `money_printed`.

<details><summary>Hint</summary>

Set $\Delta M = 0$ in equation (8.7) and solve for $\Delta B$. Then `money_printed(30, 22, ...)` should give 0.

</details>

In [50]:
# Your turn: try it here

## Figure 7: High inflation in Mexico, Nigeria, and Venezuela

**Goal.** Compare high inflation with hyperinflation, and follow Venezuela's recent hyperinflation.

In [51]:
latam = pd.read_csv(DATA_URL + "fig8-7.csv")
latam = latam.rename(columns={"mexico": "Mexico", "nigeria": "Nigeria", "venezuela": "Venezuela"})
latam.tail(9)

,year,Mexico,Nigeria,Venezuela
57,2017,6.041,16.502,438.117
58,2018,4.899,12.095,65374.100
59,2019,3.636,11.396,19906.000
60,2020,3.397,13.246,2355.150
61,2021,5.689,16.953,1588.510
62,2022,7.896,18.847,200.914
63,2023,5.528,24.660,337.458
64,2024,4.722,33.242,48.983
65,2025,3.810,23.010,NaN


The textbook's chart ends in 2016, with Venezuela at about 255 percent. Then inflation exploded: 65,374 percent in
2018 and 19,906 percent in 2019. These are not data errors. They are the hyperinflation the textbook mentions
("exceeding 10,000 percent per year"), and the World Bank's Global Inflation Database records them. A linear axis
cannot show them next to Mexico's peaks, so we use a ratio scale. Nigeria's few years of falling prices (1963, 1967
and 1968) cannot be drawn on it and leave gaps.

### Data hygiene: Zimbabwe

The menu below adds any country of the wide panel with a year above 500 percent. Before building it, look at one
country the textbook mentions, Zimbabwe, whose hyperinflation peaked in November 2008:

In [52]:
panel[(panel.country == "Zimbabwe") & (panel.year >= 2000) & (panel.year <= 2009)]

,iso3,country,year,inflation,source
10817,ZWE,Zimbabwe,2000,4.475,GID
10818,ZWE,Zimbabwe,2001,-37.198,GID
10819,ZWE,Zimbabwe,2002,-34.445,GID
10820,ZWE,Zimbabwe,2003,-8.565,GID
10821,ZWE,Zimbabwe,2004,113.569,GID
10822,ZWE,Zimbabwe,2005,-31.522,GID
10823,ZWE,Zimbabwe,2006,32.974,GID
10824,ZWE,Zimbabwe,2007,-72.729,GID
10825,ZWE,Zimbabwe,2008,156.964,GID
10826,ZWE,Zimbabwe,2009,6.216,GID


These numbers cannot be right. They say that prices **fell** by 73 percent in 2007 and rose only 157 percent in 2008,
when, according to the textbook, prices were doubling roughly every day. Zimbabwe's official price statistics broke
down during the crisis, and no reliable annual series exists. We drop Zimbabwe's values for 2001–2008. (The same years
enter Zimbabwe's average in Figure 4, so its point there is not reliable either.)

> **Python note.** `~` means "not": `panel[~bad]` keeps every row where `bad` is False.

In [53]:
bad = (panel.country == "Zimbabwe") & (panel.year >= 2001) & (panel.year <= 2008)
clean = panel[~bad]
peaks = clean.groupby("country").inflation.max()
choices = peaks[peaks > 500].index.drop("Venezuela").tolist()
print(len(choices), "countries:", choices)

30 countries: ['Angola', 'Argentina', 'Armenia', 'Azerbaijan', 'Belarus', 'Bolivia', 'Brazil', 'Bulgaria', 'Chile', 'Congo, Dem. Rep.', 'Croatia', 'Estonia', 'Georgia', 'Indonesia', 'Kazakhstan', 'Kyrgyz Republic', 'Latvia', 'Lithuania', 'Moldova', 'Nicaragua', 'North Macedonia', 'Peru', 'Poland', 'Russia', 'Slovenia', 'Tajikistan', 'Turkmenistan', 'Ukraine', 'Uzbekistan', 'Zimbabwe']


> **Python note.** `.melt(id_vars="year", ...)` turns the three country columns into rows, one row per country
> and year, so that `px.line(..., color="country")` can draw one line per country. `pd.concat([a, b])` stacks two
> tables. On a ratio axis, the position of a note is given in powers of 10, hence `np.log10(...)` in `add_annotation`.

In [54]:
RATIO_TICKS = [1, 3, 10, 30, 100, 300, 1000, 3000, 10000, 30000, 100000]


def compare_high_inflation(extra="none"):
    """Figure 8.7: inflation in Mexico, Nigeria and Venezuela (ratio scale), plus one more country."""
    long = latam.melt(id_vars="year", var_name="country", value_name="inflation")
    ven = latam.set_index("year").Venezuela
    title = f"Peak inflation: Venezuela {ven.max():,.0f}% in {ven.idxmax()}"
    if extra != "none":
        other = clean[clean.country == extra][["year", "country", "inflation"]]
        long = pd.concat([long, other])
        top = other.loc[other.inflation.idxmax()]
        title = title + f", {extra} {top.inflation:,.0f}% in {top.year:.0f}"
    long = long[long.inflation > 0]                          # a ratio scale cannot show zero or falling prices
    fig = px.line(long, x="year", y="inflation", color="country", log_y=True, hover_name="country",
                  color_discrete_sequence=[BLUE, GREEN, ORANGE, TEAL],
                  labels={"year": "Year", "inflation": "Inflation rate (percent, ratio scale)", "country": "Country"},
                  title=title)
    fig.update_traces(line=dict(width=3))
    fig.add_annotation(x=ven.idxmax(), y=np.log10(ven.max()), text=f"Venezuela, {ven.idxmax()}", showarrow=True, ax=-70, ay=10)
    fig.add_hline(y=500, line_dash="dot", line_color=GRAY, annotation_text="500%: hyperinflation",
                  annotation_position="bottom left")
    fig.update_xaxes(range=[1960, 2025])
    fig.update_yaxes(tickvals=RATIO_TICKS, range=[np.log10(0.5), np.log10(200000)])
    return fig

In [55]:
interact(compare_high_inflation, extra=Dropdown(options=["none"] + choices, value="none", description="add a country", style=WIDE));

In [56]:
# @title Website version: Figure 7 with a precomputed slider for the added country
slider_figure(compare_high_inflation, "extra", ["none", "Argentina", "Bolivia", "Nicaragua", "Peru", "Zimbabwe"],
              start="none", label="added country")

**What to notice.**

- Mexico (peaks of 102 percent in 1983 and 132 percent in 1987) and Nigeria (73 percent in 1995) had high inflation,
  but never a hyperinflation by the 500-percent definition. Both came down within a few years.
- Venezuela crossed 500 percent in 2018 and stayed above it until 2021. By 2024 inflation was down to 49 percent.
- Add Bolivia, Nicaragua or Peru: the Latin American hyperinflations of the 1980s and early 1990s reached thousands
  of percent. Bolivia's fell from its 1985 peak to 15 percent by 1987; Nicaragua's and Peru's took four or five years.

### Key numbers: Venezuela and the definition of hyperinflation

In [57]:
v = latam.set_index("year").Venezuela.dropna()
print("Venezuela:", v.loc[2016:].round(0).to_dict())
print("Years above 500%:", v.index[v > 500].tolist())
print(f"Mexico's peak: {latam.Mexico.max():.1f}% in {latam.year[latam.Mexico.idxmax()]}; "
      f"Nigeria's peak: {latam.Nigeria.max():.1f}% in {latam.year[latam.Nigeria.idxmax()]}")
print(f"Economies with at least one year above 500% (cleaned panel): {len(choices) + 1}")

Venezuela: {2016: 255.0, 2017: 438.0, 2018: 65374.0, 2019: 19906.0, 2020: 2355.0, 2021: 1589.0, 2022: 201.0, 2023: 337.0, 2024: 49.0}
Years above 500%: [2018, 2019, 2020, 2021]
Mexico's peak: 131.8% in 1987; Nigeria's peak: 72.8% in 1995
Economies with at least one year above 500% (cleaned panel): 31


The textbook (written when Venezuela's chart stopped at 255 percent) was right that a hyperinflation was under way.
In our cleaned panel, 31 economies had at least one year above 500 percent since 1960, most of them in Latin America,
in the former Soviet Union, or elsewhere in Eastern Europe.

**Your turn.** In the cleaned panel, in which year did Bolivia's inflation peak, and how many years did it take to
fall below 25 percent?

<details><summary>Hint</summary>

Keep Bolivia's rows with `clean[clean.country == "Bolivia"]`, and look at the `year` and `inflation` columns around
the peak (`.idxmax()` gives the row label of the largest value).

</details>

In [58]:
# Your turn: try it here

## Key numbers

This table collects the main numbers of the chapter. Each one is computed with the data and the functions above.

In [59]:
rows = [
    ["Quantity theory: inflation with 7% money growth and 4% GDP growth (%)", qt_inflation(7, 4)],
    ["Fisher equation: nominal rate with R = 5% and inflation 3% (%)", nominal_rate(5, 3)],
    ["U.S. inflation, 1980 peak (%)", rate[1980]],
    ["U.S. inflation, 2022 (%)", rate[2022]],
    ["U.S. price level 2025 / 1959", us.price_level.iloc[-1] / 100],
    ["27 cents of 1950 in 2022 dollars", in_dollars_of(0.27, 1950, 2022)],
    ["Countries with inflation above 25%, 1994 (%)", share_25[1994]],
    ["U.S. inflation in the 1970s (% per year)", decades.set_index("decade").inflation["1970s"]],
    ["U.S. money growth in the 1970s (% per year)", decades.set_index("decade").money_growth["1970s"]],
    ["M2 velocity, 2021", yearly.velocity[2021]],
    ["Correlation of log money growth and log inflation, 1990-2021", np.log10(world.money_growth_1990_2021).corr(np.log10(world.inflation_1990_2021))],
    ["Highest U.S. 3-month T-bill rate, 1981 (%)", r.nominal_tbill_3m[1981]],
    ["U.S. real interest rate, 2022 (%)", r.real_rate[2022]],
    ["Argentina's inflation, 1989 (%)", hyper.set_index("year").Argentina[1989]],
    ["Venezuela's inflation, 2018 (%)", v[2018]],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(2)

,Quantity,Value
0,Quantity theory: inflation with 7% money growt...,3.00
1,Fisher equation: nominal rate with R = 5% and ...,8.00
2,"U.S. inflation, 1980 peak (%)",13.50
3,"U.S. inflation, 2022 (%)",7.99
4,U.S. price level 2025 / 1959,11.05
5,27 cents of 1950 in 2022 dollars,3.28
6,"Countries with inflation above 25%, 1994 (%)",31.13
7,U.S. inflation in the 1970s (% per year),7.81
8,U.S. money growth in the 1970s (% per year),9.86
9,"M2 velocity, 2021",1.16


## Summary

- Inflation is the growth rate of the price level. U.S. inflation peaked at 13.5 percent in 1980, stayed near 2
  percent for two decades, and jumped to 8 percent in 2022. A price index turns past prices into today's dollars.
- The quantity theory of money gives $P^* = \bar{M}\bar{V}/\bar{Y}$ and $\pi^* = \bar{g}_M - \bar{g}_Y$: in the long
  run, faster money growth means higher inflation, one for one. U.S. decades and 165 economies line up as predicted,
  although velocity is not really constant.
- The Fisher equation $i = R + \pi$ turns nominal interest rates into real ones. U.S. real rates were negative in the
  1970s and in most years since 2008.
- Hyperinflations happen when governments cannot pay their bills with taxes or borrowing and print money instead:
  persistent high inflation is a fiscal problem. They can end as quickly as they start.
- The next chapter starts the short-run part of the book, with an introduction to economic fluctuations.